# ***Predictive Modeling***

To identify beneficiaries at elevated risk of healthcare cost burden, a binary classification model was developed for two outcomes: **high out-of-pocket (OOP) burden** and **high payer cost burden**. 3 modeling approaches—**logistic regression, random forest, and XGBoost**—are evaluated to compare interpretable and nonlinear machine learning methods.

The prediction framework accounts for the information that would realistically be available at each point in time:

* **2008:** Predictions use baseline beneficiary demographics and chronic disease information.
* **2009:** Predictions incorporate 2008 historical information alongside beneficiary characteristics.
* **2010:** Predictions incorporate cumulative information from **2008 and 2009**, allowing the models to leverage a longer history of beneficiary utilization, disease burden, and spending.

This temporal structure is designed to prevent information leakage by ensuring that predictors come from information available **before the outcome being predicted**.

The analysis therefore addresses two related questions:

1. **Can we identify beneficiaries who will experience high OOP or payer cost burden?**
2. **Does incorporating longitudinal beneficiary history improve our ability to identify future high-cost beneficiaries?**

Model performance is evaluated using discrimination and classification metrics, while feature-level interpretation is used to examine which demographic, clinical, utilization, and historical cost factors contribute most to high-cost predictions.


In [7]:
import numpy as np
import pandas as pd
import pickle
import os
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [9]:
patient = pd.read_csv("cleaned_data/patient_final.csv")
patient.head()

,DESYNPUF_ID,BENE_ESRD_IND,BENE_COUNTY_CD,PART_A_INSUR_CVRAGE_TOT_MONS,PART_B_INSUR_CVRAGE_TOT_MONS,HMO_INSUR_CVRAGE_TOT_MONS,PART_D_INSUR_CVRAGE_TOT_MONS,year,SEX,RACE,STATE,age,age_band,IP_bill,OP_bill,Carrier_bill,num_carrier_claims,avg_carrier_claim_duration,max_carrier_claim_duration,total_carrier_claim_duration,avg_carrier_rejected_amt,max_carrier_rejected_amt,total_carrier_rejected_amt,avg_carrier_claim_rejected_rate (%),max_carrier_claim_rejected_rate (%),avg_carrier_reimb_allowed_diff,min_carrier_reimb_allowed_diff,max_carrier_reimb_allowed_diff,total_carrier_reimb_allowed_diff,max_carrier_chronic_code_counts,avg_carrier_insurance_cost,max_carrier_insurance_cost,total_carrier_insurance_cost,avg_carrier_beneficiary_cost,max_carrier_beneficiary_cost,total_carrier_beneficiary_cost,has_Additional endocrine/metabolic burden,has_COPD,has_Administrative_screening_aftercare,has_Cancer,has_Injury/Frailty Burden,has_Additional cardiovascular burden,has_Additional mental health burden,has_Additional Musculoskeletal Burden,has_Infectious disease burden,has_Stroke/TIA,has_Additional neurologic burden,has_Diabetes,has_Additional renal burden,has_Genitourinary (non-renal) burden,has_Sensory Burden,has_Additional Hematologic burden,has_Rheumatoid Arthritis/Osteoarthritis,has_Depression,has_Heart Failure,has_CKD,has_Additional GI/Hepatic burden,num_ip_claims,avg_ip_claim_duration,max_ip_claim_duration,total_ip_claim_duration,avg_ip_pmt,max_ip_pmt,total_ip_pmt,max_ip_chronic_code_counts,avg_inpatient_insurance_cost,max_inpatient_insurance_cost,total_inpatient_insurance_cost,avg_inpatient_beneficiary_cost,max_inpatient_beneficiary_cost,total_inpatient_beneficiary_cost,num_op_claims,avg_op_claim_duration,max_op_claim_duration,total_op_claim_duration,avg_op_pmt,max_op_pmt,total_op_pmt,max_op_chronic_code_counts,avg_outpatient_insurance_cost,max_outpatient_insurance_cost,total_outpatient_insurance_cost,avg_outpatient_beneficiary_cost,max_outpatient_beneficiary_cost,total_outpatient_beneficiary_cost,num_pde_claims,total_days_medsupplied,avg_days_supplied_per_pdeclaim,max_days_supplied_per_pdeclaim,total_medoop,avg_medoop_per_claim,max_medoop_per_claim,total_raw_drug_cost,avg_raw_drug_cost_per_claim,max_raw_drug_cost_per_claim,has_carrier_record,has_ip_record,has_op_record,has_pde_record,num_service_types,ip_bill_diff,op_bill_diff,total_inpatient_insurance_cost_adj,total_outpatient_insurance_cost_adj,claim_processing_burden,payer_care_cost,out_of_pocket_care_cost,total_financial_impact,High_Payer_Cost,High_OOP_Burden
0,000102649ED5601B,0,290,12,12,0,12,2008,Female,White,VA,77,75-84,0.0,450.0,1030.0,16.0,0 days 01:30:00,1 days,1 days,0.000000,0.0,0.0,0.000000,0.0,22.500000,0.0,210.0,360.0,3.0,63.750000,290.0,1020.0,12.500000,50.0,200.0,1.0,1,1.0,1,1.0,1.0,1.0,1.0,1.0,1,1.0,1,1.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0 days 00:00:00,0 days,0 days,150.000000,320.0,450.0,1.0,120.000000,300.0,360.0,30.000000,70.0,90.0,21.0,760.0,36.190476,90.0,70.0,3.333333,10.0,500.0,23.809524,120.0,True,False,True,True,3.0,0.0,0.0,0.0,360.0,0.0,1880.0,360.0,2240.0,0,0
1,0002278C944E240A,1,250,12,12,0,12,2008,Female,White,NC,72,65-74,0.0,4120.0,6230.0,42.0,0 days 17:08:34.285714285,9 days,30 days,2.142857,50.0,90.0,4.878049,100.0,25.000000,0.0,470.0,1050.0,4.0,130.714280,750.0,5490.0,32.142857,310.0,1350.0,1.0,1,1.0,1,1.0,1.0,1.0,1.0,1.0,0,1.0,1,1.0,1.0,1.0,1.0,1,1,1,1,1.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,1 days 03:41:32.307692307,13 days,15 days,316.923077,1050.0,4120.0,7.0,199.230769,800.0,2590.0,117.692308,1000.0,1530.0,47.0,1210.0,25.744681,90.0,440.0,9.361702,170.0,3580.0,76.170213,570.0,True,False,True,True,3.0,0.0,0.0,0.0,2590.0,90.0,11660.0,3320.0,14980.0,0,1
2,000330E625C93700,0,420,12,12,0,0,2008,Female,White,AR,75,75-84,0.0,0.0,420.0,4.0,0 days 00:00:00,0 days,0 days,0.000000,0.0,0.0,0.000000,0.0,30.000000,0.0,60.0,120.0,1.0,67.500000,90.0,270.0,37.500000,70.0,150.0,0.0,0,0.0,0,0.0,1.0,0.0,

Baseline model (2008, demographics & baseline characteristic): 
BENE_ESRD_IND, BENE_COUNTY_CD, SEX, RACE, STATE, age

Prediction target: High_Payer_cost, High_OOP_Burden

Model with historical data (2009):
- demographics: BENE_ESRD_IND, BENE_COUNTY_CD, SEX, RACE, STATE, age
  
- prior clinical: 2008_max_carrier_chronic_code_counts, 2008_has_Additional endocrine/metabolic burden,	2008_has_COPD, 2008_has_Administrative_screening_aftercare, 2008_has_Cancer, 2008_has_Injury/Frailty Burden, 2008_has_Additional cardiovascular burden,	2008_has_Additional mental health burden, 2008_has_Additional Musculoskeletal Burden, 2008_has_Infectious disease burden,	2008_has_Stroke/TIA, 2008_has_Additional neurologic burden, 2008_has_Diabetes, 2008_has_Additional renal burden,	2008_has_Genitourinary (non-renal) burden, 2008_has_Sensory Burden, 2008_has_Additional Hematologic burden,	2008_has_Rheumatoid Arthritis/Osteoarthritis, 2008_has_Depression, 2008_has_Heart Failure,	2008_has_CKD, 2008_has_Additional GI/Hepatic burden, 2008_max_ip_chronic_code_counts, 2008_max_op_chronic_code_counts
  
- prior utilization: 2008_num_carrier_claims, 2008_avg_carrier_claim_duration,  2008_num_ip_claims, 2008_avg_ip_claim_duration, 2008_num_op_claims, 2008_avg_op_claim_duration, 2008_num_pde_claims, 2008_total_days_medsupplied, 2008_avg_days_supplied_per_pdeclaim

   
- prior cost: 2008_avg_carrier_rejected_amt, 2008_avg_carrier_reimb_allowed_diff, 2008_avg_carrier_insurance_cost, 2008_total_carrier_insurance_cost, 2008_avg_carrier_beneficiary_cost, 2008_total_carrier_beneficiary_cost, 2008_avg_inpatient_beneficiary_cost, 2008_total_inpatient_beneficiary_cost,
2008_avg_inpatient_insurance_cost, 2008_total_inpatient_insurance_cost_adj
2008_avg_outpatient_insurance_cost, 2008_avg_outpatient_beneficiary_cost, 2008_total_outpatient_beneficiary_cost, 2008_total_medoop, 2008_total_raw_drug_cost,	2008_total_outpatient_insurance_cost_adj, 2008_claim_processing_burden
  
- prior coverage: 2008_PART_A_INSUR_CVRAGE_TOT_MONS, 2008_PART_B_INSUR_CVRAGE_TOT_MONS,	2008_HMO_INSUR_CVRAGE_TOT_MONS, 2008_PART_D_INSUR_CVRAGE_TOT_MONS


- prior risk status (add the number of times the given patient had been labeled as high cost for payer and oop fronts. If high cost for payer in 2008, then return 1):

   prior_High_Payer_Cost_counts, prior_High_OOP_Burden_counts

  
Prediction target: High_Payer_cost, High_OOP_Burden



Model with complex historical data (2010):
- demographics: BENE_ESRD_IND, BENE_COUNTY_CD, SEX, RACE, STATE, age
  
- prior clinical (the prior prefix means 2008 and 2009. For the has_[disease] variables, if either is 1, then return 1. Else, return 0. For the chronic code counts, return the max chronic code count for 2008 and 2009 for that particular variable): prior_max_carrier_chronic_code_counts, prior_has_Additional endocrine/metabolic burden,	prior_has_COPD, prior_has_Administrative_screening_aftercare, prior_has_Cancer, prior_has_Injury/Frailty Burden, prior_has_Additional cardiovascular burden,	prior_has_Additional mental health burden, prior_has_Additional Musculoskeletal Burden, prior_has_Infectious disease burden,	prior_has_Stroke/TIA, prior_has_Additional neurologic burden, prior_has_Diabetes, prior_has_Additional renal burden,	prior_has_Genitourinary (non-renal) burden, prior_has_Sensory Burden, prior_has_Additional Hematologic burden,	prior_has_Rheumatoid Arthritis/Osteoarthritis, prior_has_Depression, prior_has_Heart Failure,	prior_has_CKD, prior_has_Additional GI/Hepatic burden, prior_max_ip_chronic_code_counts, prior_max_op_chronic_code_counts
  
- prior utilization (the cum prefix means 2008 and 2009 added together): 2009_num_carrier_claims, 2009_avg_carrier_claim_duration,  2009_num_ip_claims, 2009_avg_ip_claim_duration, 2009_num_op_claims, 2009_avg_op_claim_duration, 2009_num_pde_claims, 2009_total_days_medsupplied, 2009_avg_days_supplied_per_pdeclaim, cum_num_carrier_claims,  cum_num_ip_claims, cum_num_op_claims, cum_num_pde_claims, cum_total_days_medsupplied

   
- prior cost (the cum prefix means 2008 and 2009 added together): 2009_avg_carrier_rejected_amt, 2009_avg_carrier_reimb_allowed_diff, 2009_avg_carrier_insurance_cost, 2009_total_carrier_insurance_cost, 2009_avg_carrier_beneficiary_cost, 2009_total_carrier_beneficiary_cost, 2009_avg_inpatient_beneficiary_cost, 2009_total_inpatient_beneficiary_cost,
2009_avg_inpatient_insurance_cost, 2009_total_inpatient_insurance_cost_adj
2009_avg_outpatient_insurance_cost, 2009_avg_outpatient_beneficiary_cost, 2009_total_outpatient_beneficiary_cost, 2009_total_medoop, 2009_total_raw_drug_cost,	2009_total_outpatient_insurance_cost_adj, 2009_claim_processing_burden, cum_total_carrier_insurance_cost, cum_total_carrier_beneficiary_cost, cum_total_inpatient_beneficiary_cost, cum_total_inpatient_insurance_cost_adj, cum_total_outpatient_beneficiary_cost, cum_total_medoop, cum_total_raw_drug_cost,	cum_total_outpatient_insurance_cost_adj, cum_claim_processing_burden
  
- prior coverage (the cum prefix means 2008 and 2009 added together): 2009_PART_A_INSUR_CVRAGE_TOT_MONS, 2009_PART_B_INSUR_CVRAGE_TOT_MONS,	2009_HMO_INSUR_CVRAGE_TOT_MONS, 2009_PART_D_INSUR_CVRAGE_TOT_MONS, cum_PART_A_INSUR_CVRAGE_TOT_MONS, cum_PART_B_INSUR_CVRAGE_TOT_MONS,	cum_HMO_INSUR_CVRAGE_TOT_MONS, cum_PART_D_INSUR_CVRAGE_TOT_MONS


- prior risk status (add the number of times the given patient had been labeled as high cost for payer and oop fronts. If high cost for payer in 2008 and again in 2009, then return 2): prior_High_Payer_Cost_counts, prior_High_OOP_Burden_counts 

Prediction target: High_Payer_cost, High_OOP_Burden


In [ ]:
for year in [2008, 2009, 2010]:
    X, y = build_features(year)

    for model in models:
        fit_model(X, y, model)
        evaluate_model(...)